# Mapping the World: Clustering Countries for Smarter Tourism Insights

## Project Scope and Data Sources

The goal of this project is to cluster countries based on various travel-related industry indexes, such as hospitality and historical travel data. To achieve this, we collected data from multiple sources, ensuring a comprehensive and relevant dataset.

1. **Kaggle Tourism Dataset**  
   We sourced a portion of the data from Kaggle’s [Tourism dataset](https://www.kaggle.com/datasets/imtkaggleteam/tourism?select=26-+international-arrivals-for-personal-vs-business-and-professional-reasons.csv), which includes detailed country-level information on tourism-related metrics. The data we used focuses on key factors such as:
   - Tourist accommodation
   - Food and beverage industries
   - Economic indicators related to tourism (e.g., international arrivals and tourism expenditures)

2. **Numbeo Dataset**  
   The other half of the dataset was obtained from [Numbeo](https://www.numbeo.com/), the world’s largest cost-of-living database. According to their description, Numbeo is a crowdsourced global resource for various quality of life data, covering areas such as:
   - Cost of living
   - Housing price indicators
   - Crime rates
   - Healthcare quality
   - Transport quality  
   
   We focused on the data that was specifically related to individual countries.

By combining these two sources, we aim to create a robust framework for clustering countries based on their tourism and living quality indicators.


## Creating the final Dataset

### Creating the Final Dataset

To create the final dataset, we curated data from each source and selected only the relevant information.

1. **Numbeo Data**  
   For the data from Numbeo, we performed web scraping, which is allowed under their [usage policy](https://www.numbeo.com/). This step was necessary to collect country-specific data on various quality of life indicators.

2. **Kaggle Data**  
   For the Kaggle dataset, we encountered a challenge with historical data that wasn't always the most recent or consistent in terms of the year intervals. To address this issue, we applied an **exponentially weighted average** to the yearly data. This approach allowed us to:
   - **Consolidate multiple data points** for the same country
   - **Give more weight to recent data** while still considering the historical values, ensuring that the final dataset reflects the most accurate and up-to-date information.

By following this approach, we were able to create a balanced and accurate dataset for clustering countries based on their tourism and living quality metrics.


Imports

In [ ]:
import pandas as pd

After careful consideration of each feature, we decided to drop not relevant or too similar features. 

In [ ]:
# Scraped data from Numbeo database
numbeo_dataset = pd.read_csv("../Data/combined_numbeo_dataset_adjusted.csv")
# Selected data from Tourism dataset on Kaggle
tourism_dataset = pd.read_csv("../Data/tourism_combined_data.csv")

Merging the datasets. Because we didn't want to introduce more null data, we chose to perform an inner join even if this makes the dataset have less countries. 

In [ ]:
combined_df_inner = pd.merge(tourism_dataset,numbeo_dataset, on="Country", how="inner")
combined_df_inner 

The dataset now has data on 89 countries out of the 195 countries in the world.

In [ ]:
combined_df_inner.drop(columns=["Rent Index_cost_of_living_index", "Crime Index_crime_index", "CO2Emission Index_traffic_index"], inplace=True)

combined_df_inner.columns

: 

Renaming the columns using the same naming conventions.

In [ ]:
rename_dict = {
    'Code': 'CountryCode',
    'Country': 'CountryName',
    'tourism_employment_per_1000_over_time': 'TourismEmploymentPer1000',
    'food_employment_per_1000_over_time': 'FoodEmploymentPer1000',
    'tourism_gdp_percentage_over_time': 'TourismGDPPercentage',
    'business_to_personal_ratio_over_time': 'BusinessToPersonalRatio',
    'avg_stay_days_over_time': 'AverageStayDays',
    'inbound_arrivals_over_time': 'InboundArrivalsPer1000',
    'domestic_tourists_over_time': 'DomesticTouristsPer1000',
    'inbound_to_outbound_ratio_over_time': 'InboundToOutboundRatio',
    'Cost of Living Index_cost_of_living_index': 'CostOfLivingIndex',
    'Groceries Index_cost_of_living_index': 'GroceriesCostIndex',
    'Restaurant Price Index_cost_of_living_index': 'RestaurantPriceIndex',
    'Safety Index_crime_index': 'SafetyIndex',
    'Health Care Index_health_care_index': 'HealthCareIndex',
    'Pollution Index_pollution_index': 'PollutionIndex',
    'Quality of Life Index_quality_of_life_index': 'QualityOfLifeIndex',
    'Climate Index_quality_of_life_index': 'ClimateIndex',
    'Traffic Index_traffic_index': 'TrafficIndex',
    'Time Index(in minutes)_traffic_index': 'TrafficTimeIndexMinutes',
    'Inefficiency Index_traffic_index': 'TrafficInefficiencyIndex'
}

In [ ]:
combined_df_inner.rename(columns=rename_dict, inplace=True)

Because the majority of the columns have a 0 to positive integer scale, 0 being te lowest score and the positive integer being the highest, we decided to make all the other features increasing monotonic as well.

In [ ]:
combined_df_inner["ReverseTrafficInefficiencyIndex"] = combined_df_inner["TrafficInefficiencyIndex"].max() - combined_df_inner["TrafficInefficiencyIndex"]
combined_df_inner["ReverseTrafficTimeIndexMinutes"] = combined_df_inner["TrafficTimeIndexMinutes"].max() - combined_df_inner["TrafficTimeIndexMinutes"]
combined_df_inner["ReverseTrafficIndex"] = combined_df_inner["TrafficIndex"].max() - combined_df_inner["TrafficIndex"]
combined_df_inner["ReversePollutionIndex"] = combined_df_inner["PollutionIndex"].max() - combined_df_inner["PollutionIndex"]

Exporting the dataset.

In [ ]:
combined_df_inner.to_csv('../Data/final_dataset.csv', index=False)

## Exploratory Data Analysis

Imports

In [ ]:
import pandas as pd
import missingno
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from scipy.stats import spearmanr
import numpy as np
from sklearn.preprocessing import MinMaxScaler

from tqdm import tqdm

# Suppress FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sns.set_theme(style="white")


Importing the final dataset.

In [ ]:
df = pd.read_csv("../Data/final_dataset.csv")

### Data Quality

Looking at the distribution of null values in the dataset.

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
missingno.matrix(df)

In the worse case scenario, there are countries with as low as 7 features. The best case scenario is countries having all 25 features. 

In [ ]:
df.duplicated().sum()

No duplicate entries found.

In [ ]:
df.describe().T

Features have very different scales, however by introducing the engineered "inverse"-like columns, all of them have a increasing monotone structure and meaning.

Looking at feature distribution.

In [ ]:
numerical_columns = df.select_dtypes(include=['number']).columns

# Set up the number of rows and columns for subplots
num_features = len(numerical_columns)
num_cols = 4  
num_rows = (num_features // num_cols) + (num_features % num_cols > 0)  

# Set up the figure size
plt.figure(figsize=(15, num_rows * 4))

# Loop through numerical columns and plot distributions
for i, col in enumerate(numerical_columns, 1):
    plt.subplot(num_rows, num_cols, i)
    sns.histplot(df[col].dropna(), kde=True, bins=30)
    plt.title(col)

plt.tight_layout()
plt.show()

### Comparative Analysis

#### Safety vs Healthcare

Comparing Safety Index and Healthcare Index and looking see if they are correlated.

In [ ]:
# Select relevant columns and drop rows with NaN values in either column
subset = df[['CountryName', 'SafetyIndex', 'HealthCareIndex']].dropna()
print(subset.shape[0])

In [ ]:
# Create a clearer scatter plot with uniform dot color
plt.figure(figsize=(10, 6))
scatter_plot = sns.scatterplot(
    data=subset,
    x='HealthCareIndex',
    y='SafetyIndex',
    color='steelblue',
    edgecolor='black'
)
plt.title('Safety Index vs Health Care Index')
plt.xlabel('Health Care Index')
plt.ylabel('Safety Index')
plt.tight_layout()
plt.show()

In [ ]:
# Plot with non-linear trendline using lowess smoothing
plt.figure(figsize=(10, 6))
sns.set_theme(style="white")
nonlinear_plot = sns.regplot(
    data=subset,
    x='HealthCareIndex',
    y='SafetyIndex',
    scatter_kws={'color': 'steelblue'},
    line_kws={'color': 'darkgreen'},
    lowess=True
)
plt.title('Safety Index vs Health Care Index with Non-linear Trendline (LOWESS)')
plt.xlabel('Health Care Index')
plt.ylabel('Safety Index')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate Pearson correlation coefficient and p-value
corr_coef, p_value = spearmanr(subset['HealthCareIndex'], subset['SafetyIndex'])

# Round values
rounded_corr = round(corr_coef, 2)
rounded_p = round(p_value, 4)

# Interpret correlation strength with buffer ranges
if abs(rounded_corr) < 0.28:
    strength = "weak"
elif abs(rounded_corr) < 0.33:
    strength = "borderline weak-to-moderate"
elif abs(rounded_corr) < 0.57:
    strength = "moderate"
elif abs(rounded_corr) < 0.63:
    strength = "borderline moderate-to-strong"
else:
    strength = "strong"

# Interpret significance
if rounded_p < 0.05:
    significance = "statistically significant"
else:
    significance = "not statistically significant"

# Output
print(f"Pearson Correlation Coefficient: {rounded_corr}")
print(f"→ This indicates a {strength} positive correlation between Health Care Index and Safety Index.")

print(f"\nP-value: {rounded_p}")
print(f"→ Since it's {'below' if rounded_p < 0.05 else 'above'} 0.05, this correlation is {significance}.")

In [ ]:
# Rank the data
ranked_data = subset[['CountryName', 'HealthCareIndex', 'SafetyIndex']].copy()
ranked_data['HealthCareRank'] = ranked_data['HealthCareIndex'].rank(ascending=False)
ranked_data['SafetyRank'] = ranked_data['SafetyIndex'].rank(ascending=False)
ranked_data['RankDifference'] = abs(ranked_data['HealthCareRank'] - ranked_data['SafetyRank'])

# Sort by difference for clarity in plotting
ranked_data_sorted = ranked_data.sort_values(by='RankDifference', ascending=False)

In [ ]:
# Recalculate "Same Rank" using a buffer of ±5
ranked_data['HigherRankingBuffered'] = np.where(
    abs(ranked_data['HealthCareRank'] - ranked_data['SafetyRank']) <= 5,
    'Ranks Roughly Equal',
    np.where(
        ranked_data['HealthCareRank'] < ranked_data['SafetyRank'],
        'Health Care Ranked Higher',
        'Safety Ranked Higher'
    )
)

# Sort for plotting
ranked_data_sorted = ranked_data.sort_values(by='RankDifference', ascending=False)

# Plot with buffered interpretation of rank similarity
plt.figure(figsize=(14, 10))
sns.set_theme(style="whitegrid")
barplot = sns.barplot(
    data=ranked_data_sorted,
    x='RankDifference',
    y='CountryName',
    hue='HigherRankingBuffered',
    dodge=False,
    palette={
        'Health Care Ranked Higher': 'skyblue',
        'Safety Ranked Higher': 'salmon',
        'Ranks Roughly Equal': 'gray'
    },
    edgecolor='black'
)
plt.title('Rank Differences Between Health Care and Safety Indices (with ±5 Buffer)')
plt.xlabel('Absolute Rank Difference')
plt.ylabel('Country')
plt.legend(title='Relative Ranking')
plt.tight_layout()
plt.show()



In [ ]:
# Sort by the original absolute rank difference (as in the rank difference chart)
ranked_data_sorted = ranked_data.sort_values(by='RankDifference', ascending=True)

# Plot mirrored bar chart using raw index values (no normalization)
plt.figure(figsize=(14, 12))
sns.set_theme(style="whitegrid")

# Left bars (Health Care Index) as negative
plt.barh(
    ranked_data_sorted['CountryName'],
    -ranked_data_sorted['HealthCareIndex'],
    color='skyblue',
    edgecolor='black',
    label='Health Care Index'
)

# Right bars (Safety Index)
plt.barh(
    ranked_data_sorted['CountryName'],
    ranked_data_sorted['SafetyIndex'],
    color='salmon',
    edgecolor='black',
    label='Safety Index'
)

# Update tick labels to show as positive, even on the negative side
xrange =range(-100, 101, 5)
plt.xticks(
    ticks=xrange,
    labels=[str(abs(x)) for x in xrange]
)


plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Index Value (0 to 100)')
plt.title('Health Care vs Safety Index (Mirrored Bar Chart, Sorted by Rank Difference)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


#### Climate vs Quality vs Safety vs Pollution

Next, I'll create a hexbin plot to examine the relationship between climate, quality of life, and safety. Visualizing this data allows for easier detection of trends and outliers among countries.

In [ ]:
subset = df[['CountryName', 'ClimateIndex', 'QualityOfLifeIndex', 'SafetyIndex', 'PollutionIndex']].dropna()

scaler = MinMaxScaler(feature_range=(0, 100))
subset['QualityOfLifeIndex'] = scaler.fit_transform(subset[['QualityOfLifeIndex']])

data = subset[['CountryName','ClimateIndex','QualityOfLifeIndex', 'SafetyIndex']]


plt.figure(figsize=(15, 8))
hb = plt.hexbin(
    data['ClimateIndex'], 
    data['QualityOfLifeIndex'], 
    data['SafetyIndex'],
    gridsize=25, cmap='coolwarm', alpha=0.6
)

for i, row in data.iterrows():
    plt.text(row['ClimateIndex'], row['QualityOfLifeIndex'], row['CountryName'], fontsize=8, ha='center', alpha=0.7)

plt.colorbar(hb, label="Safety Index")
plt.xlabel("Climate Index")
plt.ylabel("Quality of Life Index")
plt.title("Hexbin Plot: Climate vs Quality of Life with Country Labels")
plt.grid(True)
plt.show()


The hexbin plot shows that countries with higher climate and quality of life scores often also have higher safety levels. Warm-colored hexagons highlight regions where safety is strongest, typically aligning with better climate and quality of life. The plot also reveals some outliers, where countries deviate from this trend. Overall, it supports the idea that good climate and quality of life often go hand-in-hand with greater safety.

Now I'm going to plot a heatmap to compare Climate Index and Quality of Life Index across countries. This visual will help identify which countries score high or low in these areas, making it easier to spot patterns, outliers, and regional trends.

In [ ]:
heatmap_data = subset[['CountryName', 'ClimateIndex', 'QualityOfLifeIndex']]

heatmap_data.set_index('CountryName', inplace=True)

plt.figure(figsize=(15, 20))
sns.heatmap(
    heatmap_data, 
    cmap="coolwarm", 
    annot=True,  
    fmt=".1f",   
    linewidths=0.5
)

plt.title("Heatmap: Climate vs Quality of Life by Country", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()


The heatmap offers a clear visual comparison of the Climate Index and Quality of Life Index across different countries. By using a color gradient, it highlights where countries stand in terms of these two key indicators. Warmer colors indicate higher scores, revealing which countries enjoy both favorable climates and a high quality of life, while cooler tones point to nations with lower performance in one or both areas. The inclusion of exact values makes it easy to identify patterns, outliers, and regional differences at a glance. Overall, the heatmap effectively summarizes complex data, making it easier to interpret and compare countries based on their climate and living conditions.

Now I'm going to plot a hexbin chart to examine the relationship between climate, quality of life, and pollution across countries. This will help visualize how environmental factors like pollution interact with living conditions, and whether better climate and quality of life are consistently associated with lower pollution levels.

In [ ]:
data = subset[['CountryName','ClimateIndex','QualityOfLifeIndex', 'PollutionIndex']]

plt.figure(figsize=(12, 8))
hb = plt.hexbin(
    data['ClimateIndex'], 
    data['QualityOfLifeIndex'], 
    data['PollutionIndex'],
    gridsize=25, cmap='coolwarm', alpha=0.6
)

for i, row in data.iterrows():
    plt.text(row['ClimateIndex'], row['QualityOfLifeIndex'], row['CountryName'], fontsize=8, ha='center', alpha=0.7)

plt.colorbar(hb, label="Pollution")
plt.xlabel("Climate Index")
plt.ylabel("Quality of Life Index")
plt.title("Hexbin Plot: Climate vs Quality of Life with Country Labels")
plt.grid(True)
plt.show()


This hexbin plot illustrates the relationship between Climate Index and Quality of Life Index, with color intensity representing the Pollution Index across countries. The visualization reveals that countries with a better climate often tend to have a higher quality of life. However, the color shading shows varying pollution levels within these clusters—some countries with high climate and quality of life scores still experience elevated pollution, while others maintain low pollution levels. This suggests that while climate and quality of life are positively related, pollution does not always follow the same trend and can vary significantly. Overall, the plot helps identify countries where favorable living conditions may be offset by environmental concerns.

#### Safety vs Average Stay

We will compare the two features safety and average stay to check if they are correlated.

In [ ]:
subset = df[['CountryName', 'SafetyIndex', 'AverageStayDays']].dropna()
scaler = MinMaxScaler(feature_range = (0,100))
subset[['ScaledSafetyIndex', 'ScaledAverageStay']] = scaler.fit_transform(subset[['SafetyIndex', 'AverageStayDays']])

In [ ]:
# %%
# Scatter plot with scaled data
plt.figure(figsize=(12, 8))

sns.scatterplot(x=subset['ScaledAverageStay'], y=subset['ScaledSafetyIndex'])
plt.title('Scaled Average Stay Days vs Scaled Safety Index')
plt.xlabel('Scaled Average Stay Days')
plt.ylabel('Scaled Safety Index')
plt.show()


The scatter plot shows the relationship between scaled average stay days and the scaled safety index. We observe that most data points cluster at lower average stay days (0-20), regardless of safety index. A few data points have higher average stay days (up to 100), and these correspond to lower safety index values. This suggests a possible inverse relationship: as the safety index decreases, average stay days tend to increase.

In [ ]:
# %%
# Correlation heatmap (scaled)
corr = subset[['ScaledAverageStay', 'ScaledSafetyIndex']].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Scaled): Average Stay vs Safety Index')
plt.show()


The heatmap displays the correlation between scaled average stay days and scaled safety index. The correlation coefficient of -0.26 suggests a weak negative correlation: as the scaled safety index decreases, scaled average stay days tend to increase slightly. However, the strength of this relationship is modest, indicating that other factors may also influence the average stay days.



In [ ]:
# Correlation between scaled variables
correlation = subset['ScaledAverageStay'].corr(subset['ScaledSafetyIndex'])
print('Correlation between Scaled Average Stay and Scaled Safety Index:', correlation)

The analysis found a weak negative correlation (-0.26) between safety and average stay duration, indicating that countries with higher safety indexes tend to have slightly shorter average stays. Visualizations confirmed this subtle inverse relationship but suggest other factors also influence stay length.

In [ ]:
# %%
# Country-wise heatmap using scaled values
heatmap_data = subset[['CountryName', 'ScaledAverageStay', 'ScaledSafetyIndex']].copy()
heatmap_data.set_index('CountryName', inplace=True)

plt.figure(figsize=(15, 20))
sns.heatmap(
    heatmap_data,
    cmap="coolwarm",
    annot=True,
    fmt=".2f",
    linewidths=1.5
)
plt.title("Scaled Average Stay Days vs Scaled Safety Index by Country", fontsize=14)
plt.show()


This heatmap presents scaled average stay days (left) and scaled safety index (right) for various countries. While the scaled stay days vary significantly, some countries with high average stay days (like Costa Rica, Egypt, and El Salvador) also have relatively low safety indexes, suggesting a possible inverse relationship. However, many countries do not follow this pattern, indicating the relationship might be influenced by other regional or socioeconomic factors.



## Data Preparation

In [ ]:
df = pd.read_csv("../Data/final_dataset.csv")
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe().T

In [ ]:
all_numerical_features_df = df.copy()
all_numerical_features_df.drop(columns=["CountryName","CountryCode", "TrafficIndex", "TrafficTimeIndexMinutes","TrafficInefficiencyIndex", "PollutionIndex"], inplace=True)

### Imputation

We will use maximum-likelihood estimation (MLE) for imputing all missing data, a multivariate imputation approach.

In [ ]:
from fancyimpute import IterativeImputer

imputer = IterativeImputer(max_iter=100, random_state=0)

imputed_data = imputer.fit_transform(all_numerical_features_df)

imputed_df = pd.DataFrame(imputed_data, columns=all_numerical_features_df.columns)

print(imputed_df.head())

In [ ]:
imputed_df.isnull().sum()

### Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data=scaler.fit_transform(imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=imputed_df.columns)

default_dataset = pd.concat([df[["CountryName", "CountryCode"]], scaled_df] , axis=1)

default_dataset.head()

## Clustering

### Outlier Detection and Clustering

Before proceeding with clustering, we first experimented with outlier detection algorithms to identify any anomalous data points.

1. **DBScan**  
   We applied **DBScan** (Density-Based Spatial Clustering of Applications with Noise), which is effective for identifying outliers in data that has varying densities. This algorithm helps in detecting regions of high data density while classifying low-density points as outliers.

2. **K-Means Clustering**  
   We also tested **K-Means clustering** to determine whether the data naturally divides into distinct clusters. Although K-Means is more commonly used for clustering, we utilized it here as an additional technique to assess potential outliers based on the distances to the nearest cluster centers.


In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import cdist, pdist
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, pairwise_distances
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")
 
import utils

In [ ]:
df = pd.read_csv("../Data/final_cleaned_dataset.csv", index_col=0)

labels_df = df[['CountryCode', 'CountryName']]
numerical_features_df = df.drop(columns=['CountryCode', 'CountryName'])

#### K-Means 

In [ ]:
number_clusters = range(2, 11) #between 2 and 11 clusters
silhouette_scores, inertia = utils.getSilhouetteScoreAndInertiaForKMeans(number_clusters, numerical_features_df)
utils.plotValuesBasedOnClusterNumbers(number_clusters, silhouette_scores, "Number of Clusters", "Silhouette Score","Silhouette Score vs Number of Clusters")
utils.plotValuesBasedOnClusterNumbers(number_clusters, inertia, "Number of Clusters", "Inertia","Elbow method vs Number of Clusters")

#Choose the best k (based on max silhouette score)
best_k = number_clusters[inertia.index(max(inertia))]

#Run K-Means
kmeans_final = KMeans(n_clusters=best_k,random_state=43, n_init=10, max_iter=1000)
cluster_labels = kmeans_final.fit_predict(numerical_features_df)


# Get distances of each point to its assigned cluster centroid
centroids = kmeans_final.cluster_centers_
distances = np.linalg.norm(numerical_features_df - centroids[cluster_labels], axis=1)

# Define outliers (e.g., top 5% farthest points)
threshold = np.percentile(distances, 95)
outliers = distances > threshold

# Run PCA
pca = PCA(n_components=2)
pca_components = pca.fit_transform(numerical_features_df)

plt.figure(figsize=(10, 6))

# Color points: red for outliers, blue otherwise
colors = ['red' if outliers[i] else 'blue' for i in range(len(outliers))]
plt.scatter(pca_components[:, 0], pca_components[:, 1], c=colors, s=80, edgecolors='k')

# Annotate country names using same red/blue logic
for i, country in enumerate(df["CountryName"]):
    label_color = 'red' if outliers[i] else 'blue'
    plt.annotate(country, (pca_components[i, 0], pca_components[i, 1]), fontsize=8, color=label_color)

plt.title("Outlier Detection via K-Means (Red = Outliers, Blue = Inliers)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.show()

# Print names of outlier countries
outlier_countries = df["CountryName"][outliers]
print("Outlier Countries:")
print(outlier_countries.to_list())

Validate Clustering Quality:
We compute the following **validation metrics** to assess the quality of clustering:
- **Davies-Bouldin Index** (lower is better)
- **Calinski-Harabasz Index** (higher is better)
- **Dunn Index** (higher is better)

In [ ]:
results_df = pd.DataFrame(columns=[
    "Dataset",
    "Algorithm",
    "Davies-Bouldin Index",
    "Calinski-Harabasz Index",
    "Dunn Index"
])

: 

### Baseline

## Conclusions

## SWOT Analysis

## Strengths: 

Comprehensive data exploration: The approach starts with thorough EDA, understanding missing data, distributions, and feature relationships. 
Multi-step data cleaning and feature handling: From imputation to scaling,  systematically prepared data for clustering. 
Statistical rigor: Incorporating correlation heatmaps, permutation tests, and rank-based comparisons adds validity to the insights. 
Use of multiple clustering techniques: K-Means, Agglomerative Clustering, and DBSCAN provide multiple lenses to identify patterns, ensuring no single algorithm biases the results. 
Visualization-driven approach: The methodology prioritizes visual validation (e.g., PCA, t-SNE, dendrograms) to confirm and interpret cluster separation. 

  
## Weaknesses: 

Small dataset: With only 89 entries, the statistical power is limited, making clusters more susceptible to noise. 
Potential overinterpretation of clusters: Clustering is an unsupervised technique; interpreting clusters as “truth” without domain knowledge context can be misleading. 
Limited real-world profiling of clusters: The methodology clusters the data well, but does not interpret clusters in the broader context (e.g., economic factors, regional stability). 

 
## Opportunities: 

Expand the data: More data or additional features (e.g., economic, cultural, political data) would provide a richer basis for clustering. 
Test multiple imputation strategies: Try simpler imputation (mean/median) vs. MLE to see if cluster stability changes. 
Develop interpretive narratives: Create profiles of each cluster to explain what they represent, improving stakeholder understanding. 
Interactive presentation: Present the methodology interactively (dashboards) for dynamic interpretation and easier updates. 

 
## Threats: 

Sensitivity of clustering algorithms: Clustering depends on distance measures and scaling. Small changes in preprocessing can change results significantly. 
Data quality risks: High missingness in some features may lead to misleading clusters even with sophisticated imputation. 
Potential for overfitting: Complex methodology on a small dataset can overfit patterns that don’t generalize. 
Static insights: Without dynamic updates, static methodology might become outdated as data changes.